In [1]:
import numpy as np
import pandas as pd

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [3]:
!wget $data -O data-week-3-homework.csv

--2026-10-12 18:00:17--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘data-week-3-homework.csv’

data-week-3-homewor 100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-10-12 18:00:17 (107 MB/s) - ‘data-week-3-homework.csv’ saved [278746/278746]



In [4]:
df = pd.read_csv('data-week-3-homework.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [7]:
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

In [8]:
categorical_columns

['lead_source', 'industry', 'employment_status', 'location']

In [9]:
numerical_columns = list(df.dtypes[(df.dtypes == 'float64') | (df.dtypes == 'int64')].index)
numerical_columns

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [10]:
df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [11]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# Q1

In [12]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

# Q2

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [15]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [16]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [17]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


In [18]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [19]:
df_full_train[numerical_columns].corrwith(df_full_train.converted)

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
converted                   1.000000
dtype: float64

In [20]:
features = [c for c in numerical_columns if c != 'converted']

In [21]:
corr_matrix = df_full_train[features].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.163680,0.122528,0.222493
number_of_courses_viewed,0.163680,1.000000,0.723040,0.757178
interaction_count,0.122528,0.723040,1.000000,0.916626
lead_score,0.222493,0.757178,0.916626,1.000000


In [22]:
corr_matrix.loc['interaction_count', 'lead_score']

np.float64(0.9166258153290465)

In [23]:
corr_matrix.loc['number_of_courses_viewed', 'lead_score']

np.float64(0.7571778868541067)

In [24]:
corr_matrix.loc['number_of_courses_viewed', 'interaction_count']

np.float64(0.7230400573722855)

In [25]:
corr_matrix.loc['annual_income', 'interaction_count']

np.float64(0.12252817106738953)

# Q3

In [26]:
from sklearn.metrics import mutual_info_score

In [27]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [28]:
df_train[categorical_columns].head()

,lead_source,industry,employment_status,location
0,paid_ads,finance,employed,asia
1,events,finance,employed,south_america
2,referral,NA,employed,asia
3,organic_search,finance,employed,north_america
4,events,technology,employed,north_america


In [29]:
score = df_train[categorical_columns].apply(mutual_info_converted_score)
round(score.sort_values(ascending=False), 2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [30]:
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00
...,...,...,...,...,...,...,...,...
2995,paid_ads,finance,employed,asia,81522.0,1,3,0.29
2996,social_media,retail,employed,asia,33075.0,1,1,0.30
2997,organic_search,finance,employed,africa,71532.0,3,5,0.58
2998,social_media,retail,employed,asia,48836.0,0,0,0.03


# Q4

In [31]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [32]:
train_dicts = df_train[categorical_columns + features].to_dict(orient='records')

In [33]:
dv = DictVectorizer(sparse=False)

In [34]:
X_train = dv.fit_transform(train_dicts)

In [35]:
val_dicts = df_val[categorical_columns + features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [36]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [37]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [38]:
y_pred = model.predict(X_val)

In [39]:
accuracy = (y_val == y_pred).mean()
round(accuracy, 2)

np.float64(0.64)

# Q5

In [40]:
features

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [41]:
def feature_elimination(features):
    train_dicts = df_train[features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = (y_val == y_pred).mean()

    return accuracy

In [44]:
all_features = categorical_columns + features

original_accuracy = feature_elimination(all_features)

In [47]:
accuracies = {}

for name in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    subset = [c for c in all_features if c != name]
    accuracies[name] = feature_elimination(subset)

for k,v in accuracies.items():
    print(k, original_accuracy - v)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


# Q6

In [50]:
accuracies = {}

train_dicts = df_train[all_features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[all_features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = (y_val == y_pred).mean()

    accuracies[c] = accuracy

for k, v in accuracies.items():
    print(k, ':', round(v, 3))


1e-06 : 0.598
1e-05 : 0.598
0.0001 : 0.613
0.001 : 0.645
